# E-Commerce Sales Analysis & Customer Segmentation (RFM)

**Business problem:** An online retailer wants to know (1) which products, countries and months drive its revenue, and (2) who its best customers are, so it can run targeted loyalty and win-back campaigns instead of sending the same offer to everyone.

**Dataset:** *Online Retail* (UCI / Kaggle), about 540,000 transactions from a UK-based online store (Dec 2010 to Dec 2011). Save the CSV as `online_retail.csv` next to this notebook.
Columns: `InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country`.

**Tools:** Python (pandas, NumPy, Matplotlib, Seaborn, scikit-learn), SQL (SQLite).

**Workflow:** 1. Load -> 2. Clean -> 3. SQL analysis -> 4. Exploratory analysis -> 5. RFM segmentation -> 6. (Bonus) K-Means -> 7. Recommendations

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format
os.makedirs("outputs", exist_ok=True)   # charts and result files are saved here

ModuleNotFoundError: No module named 'numpy'

## 1. Load the data
First look: size, columns, data types and how much is missing. Never analyse before you know what is in the data.

In [ ]:
df = pd.read_csv("online_retail.csv", encoding="ISO-8859-1")
print("Shape:", df.shape)
display(df.head())
df.info()
print("\nShare of missing values per column:")
display(df.isna().mean().round(3))

## 2. Data cleaning
| Problem | Why it matters | Fix |
|---|---|---|
| Missing `CustomerID` | RFM is calculated per customer, so unknown customers cannot be segmented | Drop those rows |
| Invoices starting with **C** | These are cancellations and would reduce revenue wrongly | Drop |
| Negative or zero quantity / price | Returns, damaged stock or entry errors | Drop |
| Duplicate rows | Double count revenue | Drop |
| `InvoiceDate` stored as text | Cannot do date maths | Convert to datetime |

I also create a `Revenue = Quantity x UnitPrice` column, because the raw data has no revenue column.

In [ ]:
raw_rows = len(df)

df = df.dropna(subset=["CustomerID"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df.drop_duplicates()

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int)
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

removed = raw_rows - len(df)
print(f"Rows before: {raw_rows:,} | after: {len(df):,} | removed: {removed:,} ({removed/raw_rows:.1%})")
print("Date range:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())

# Headline business KPIs
total_revenue = df["Revenue"].sum()
orders = df["InvoiceNo"].nunique()
customers = df["CustomerID"].nunique()
print(f"\nTotal revenue : {total_revenue:,.0f}")
print(f"Orders        : {orders:,}")
print(f"Customers     : {customers:,}")
print(f"Average order value (AOV): {total_revenue/orders:,.2f}")

## 3. SQL analysis
Analysts usually pull data from databases with SQL, so I load the clean data into SQLite and answer business questions with queries.
Dates are stored as ISO text (`YYYY-MM-DD HH:MM:SS`) because SQLite's `strftime` needs that format.

In [ ]:
conn = sqlite3.connect(":memory:")
sql_df = df.copy()
sql_df["InvoiceDate"] = sql_df["InvoiceDate"].dt.strftime("%Y-%m-%d %H:%M:%S")
sql_df.to_sql("sales", conn, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql(sql, conn)

# Q1. Monthly revenue and number of orders
monthly = q("""
SELECT strftime('%Y-%m', InvoiceDate) AS month,
       ROUND(SUM(Revenue), 2)         AS revenue,
       COUNT(DISTINCT InvoiceNo)      AS orders
FROM sales
GROUP BY month
ORDER BY month;
""")
display(monthly)

In [ ]:
# Q2. Month-over-month revenue growth (window function LAG)
mom = q("""
WITH m AS (
    SELECT strftime('%Y-%m', InvoiceDate) AS month, SUM(Revenue) AS revenue
    FROM sales GROUP BY month
)
SELECT month,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM m
ORDER BY month;
""")
display(mom)

# Q3. Top 10 products by revenue
top_products = q("""
SELECT Description, ROUND(SUM(Revenue), 2) AS revenue, SUM(Quantity) AS units_sold
FROM sales
GROUP BY Description
ORDER BY revenue DESC
LIMIT 10;
""")
display(top_products)

# Q4. Top 10 countries by revenue, with number of customers
by_country = q("""
SELECT Country,
       ROUND(SUM(Revenue), 2)       AS revenue,
       COUNT(DISTINCT CustomerID)   AS customers
FROM sales
GROUP BY Country
ORDER BY revenue DESC
LIMIT 10;
""")
display(by_country)

## 4. Exploratory analysis (charts)
Charts turn the query results into something a manager understands in 5 seconds.
*Note: the last month in this dataset is usually incomplete, so a drop there is not a real decline.*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly["month"], monthly["revenue"], marker="o")
ax.set_title("Monthly revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("outputs/monthly_revenue.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=top_products, y="Description", x="revenue", color="steelblue", ax=axes[0])
axes[0].set_title("Top 10 products by revenue")
axes[0].set_ylabel("")

sns.barplot(data=by_country, y="Country", x="revenue", color="seagreen", ax=axes[1])
axes[1].set_title("Top 10 countries by revenue")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("outputs/top_products_countries.png", dpi=150)
plt.show()

In [ ]:
# When do customers buy? Revenue by weekday and by hour
df["Weekday"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df.groupby("Weekday")["Revenue"].sum().reindex(order).fillna(0).plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Revenue by weekday")
df.groupby("Hour")["Revenue"].sum().plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Revenue by hour of day")
plt.tight_layout()
plt.savefig("outputs/weekday_hour.png", dpi=150)
plt.show()

# Pareto check: how much revenue do the top 20% of customers bring?
cust_rev = df.groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False)
top20_share = cust_rev.head(int(len(cust_rev) * 0.2)).sum() / cust_rev.sum()
print(f"Top 20% of customers generate {top20_share:.1%} of revenue")

## 5. RFM customer segmentation (core of the project)
RFM scores every customer on three behaviours:

* **Recency (R):** days since the last purchase. **Lower is better.**
* **Frequency (F):** number of distinct orders. Higher is better.
* **Monetary (M):** total money spent. Higher is better.

**Steps:** (1) pick a "today" date one day after the last transaction, (2) calculate R, F, M per customer, (3) split each into 4 equal groups (quartiles) and score 1 to 4, where 4 is best (so Recency labels are reversed), (4) add the three scores (range 3 to 12), (5) map to named segments.

I use `rank(method="first")` before `qcut` because many customers share the same Frequency (for example, one order), and `qcut` fails when bin edges repeat.

In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
)

rfm["R"] = pd.qcut(rfm["Recency"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["Score"] = rfm["R"] + rfm["F"] + rfm["M"]

def segment(row):
    if row["R"] >= 3 and row["Score"] >= 10:
        return "Champions"            # bought recently, often, spend a lot
    elif row["R"] <= 2 and row["F"] >= 3:
        return "At Risk"              # used to buy often, gone quiet
    elif row["Score"] >= 8:
        return "Loyal"
    elif row["Score"] >= 6:
        return "Potential Loyalists"
    else:
        return "Lost / Low Value"

rfm["Segment"] = rfm.apply(segment, axis=1)
display(rfm.head(10))

In [ ]:
seg = rfm.groupby("Segment").agg(
    customers=("Monetary", "size"),
    avg_recency_days=("Recency", "mean"),
    avg_orders=("Frequency", "mean"),
    avg_spend=("Monetary", "mean"),
    total_revenue=("Monetary", "sum"),
)
seg["pct_customers"] = 100 * seg["customers"] / seg["customers"].sum()
seg["pct_revenue"] = 100 * seg["total_revenue"] / seg["total_revenue"].sum()
seg = seg.sort_values("total_revenue", ascending=False)
display(seg.round(1))

In [ ]:
# Share of customers vs share of revenue per segment: the key chart for the business story
comp = seg[["pct_customers", "pct_revenue"]]
ax = comp.plot(kind="bar", figsize=(10, 5), color=["#9ecae1", "#08519c"])
ax.set_ylabel("Percent")
ax.set_title("Share of customers vs share of revenue by segment")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("outputs/segment_share.png", dpi=150)
plt.show()

plt.figure(figsize=(9, 6))
sns.scatterplot(data=rfm, x="Frequency", y="Monetary", hue="Segment", alpha=0.6)
plt.xscale("log"); plt.yscale("log")
plt.title("Customers by Frequency and Monetary value (log scale)")
plt.tight_layout()
plt.savefig("outputs/rfm_scatter.png", dpi=150)
plt.show()

## 6. Bonus: K-Means clustering as a cross-check
Rule-based RFM is easy to explain. K-Means lets the **data** find the groups. Because RFM values are heavily right-skewed, I apply `log1p` and then standardise so no single feature dominates the distance calculation. The elbow plot helps choose the number of clusters.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

X = np.log1p(rfm[["Recency", "Frequency", "Monetary"]])
Xs = StandardScaler().fit_transform(X)

inertia = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs).inertia_ for k in range(1, 9)]
plt.figure(figsize=(6, 4))
plt.plot(range(1, 9), inertia, marker="o")
plt.xlabel("Number of clusters (k)"); plt.ylabel("Inertia"); plt.title("Elbow method")
plt.tight_layout()
plt.savefig("outputs/elbow.png", dpi=150)
plt.show()

km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xs)
rfm["Cluster"] = km.labels_
display(rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].agg(["mean", "count"]).round(1))

## 7. Export results

In [ ]:
rfm.to_csv("outputs/rfm_customer_segments.csv")
seg.to_csv("outputs/segment_summary.csv")
monthly.to_csv("outputs/monthly_revenue.csv", index=False)
print("Saved to the outputs/ folder:", os.listdir("outputs"))

## 8. Business recommendations
| Segment | Who they are | Suggested action |
|---|---|---|
| **Champions** | Recent, frequent, high spenders | Loyalty rewards, early access to new products, ask for reviews and referrals. Do not discount heavily; they already buy. |
| **Loyal** | Good scores, steady buyers | Upsell and cross-sell, bundle offers |
| **Potential Loyalists** | Mid scores, fairly recent | Nudge to a second or third purchase with a small incentive |
| **At Risk** | Used to buy often, silent now | Win-back campaign (personalised email, time-limited offer). Highest priority because they were valuable. |
| **Lost / Low Value** | Old, one-time, low spend | Low-cost reactivation email only; do not spend much budget |

## 9. My findings (fill in after running; these are the numbers for your resume and interview)
* Rows cleaned: ____ removed out of ____ (____%)
* Peak revenue month: ____ ; revenue growth that month: ____%
* Top product: ____ ; top country: ____ with ____% of revenue
* Top 20% of customers generate ____% of revenue
* Champions are ____% of customers but ____% of revenue
* At Risk customers: ____ (potential revenue to win back: ____)

## 10. Limitations and next steps
* One year of data from one retailer, so seasonality cannot be separated from a trend.
* RFM thresholds (quartiles and the segment rules) are a business choice, not a law; they can be tuned.
* Next steps: add a customer lifetime value (CLV) estimate, test the campaigns with an A/B test, and build a Power BI dashboard on `outputs/rfm_customer_segments.csv`.